# Data Wrangling — Vagas em Ciência de Dados

**Autor:** Felipe Rossigalli Ianelo
**Contato:** [LinkedIn](https://www.linkedin.com/in/felipe-ianelo/) · [GitHub](https://github.com/rfianelo)

---

## Objetivo do script

Este script realiza a limpeza e o tratamento (data wrangling) de uma base de
vagas de emprego na área de ciência de dados, extraída do Glassdoor. O objetivo
é transformar dados brutos e ruidosos em uma base estruturada e
consistente, pronta para análise e visualização.

Ao longo do notebook são tratados os campos de cargo, senioridade, salário,
empresa, localização e porte, além do tratamento de valores faltantes,
resultando em uma base final limpa exportada em formato CSV.

## Fonte dos dados
Dataset *"Uncleaned DS Jobs"* (Glassdoor), disponível no Kaggle:
`rashikrahmanpritom/data-science-job-posting-on-glassdoor`

## Ferramentas
Python (pandas, numpy) · Google Colab

In [2]:
import pandas as pd
import numpy as np
import kagglehub
import os

In [3]:
path = kagglehub.dataset_download("rashikrahmanpritom/data-science-job-posting-on-glassdoor")
dfjobs = pd.read_csv(os.path.join(path, "Uncleaned_DS_jobs.csv"))


100%|██████████| 1.53M/1.53M [00:00<00:00, 101MB/s]

Extracting files...


In [4]:
dfjobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 672 entries, 0 to 671
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   index              672 non-null    int64  
 1   Job Title          672 non-null    object 
 2   Salary Estimate    672 non-null    object 
 3   Job Description    672 non-null    object 
 4   Rating             672 non-null    float64
 5   Company Name       672 non-null    object 
 6   Location           672 non-null    object 
 7   Headquarters       672 non-null    object 
 8   Size               672 non-null    object 
 9   Founded            672 non-null    int64  
 10  Type of ownership  672 non-null    object 
 11  Industry           672 non-null    object 
 12  Sector             672 non-null    object 
 13  Revenue            672 non-null    object 
 14  Competitors        672 non-null    object 
dtypes: float64(1), int64(2), object(12)
memory usage: 78.9+ KB


In [5]:
# Substituindo os valores faltantes, codificados como -1 na base original, por NaN. Isso garante que esses valores não vão interir na análise de variáveis numéricas.
dfjobs = dfjobs.replace([-1, '-1'], np.nan)

In [6]:
# Criando a coluna Senioridade a partir da descrição dos cargos. Foi utilizando um dicionário de pré-fixos frequentemente utilizados em descrições de vaga para a identificação a partir da coluna 'Job Title'.
title_n = dfjobs['Job Title'].str.lower()

condicoes = [
    title_n.str.contains('jr|junior|entry|associate'),
    title_n.str.contains('sr|senior'),
    title_n.str.contains('lead'),
    title_n.str.contains('principal|staff'),
    title_n.str.contains('manager'),
    title_n.str.contains('head'),
    title_n.str.contains('director|vp|chief'),
]

valores = ['Junior', 'Senior', 'Lead', 'Principal', 'Manager', 'Head', 'Director']

dfjobs['Senioridae']= np.select(condicoes, valores, 'Unknown')

In [7]:
dfjobs['Job Title'].unique()

array(['Sr Data Scientist', 'Data Scientist',
       'Data Scientist / Machine Learning Expert',
       'Staff Data Scientist - Analytics',
       'Data Scientist - Statistics, Early Career', 'Data Modeler',
       'Experienced Data Scientist', 'Data Scientist - Contract',
       'Data Analyst II', 'Medical Lab Scientist',
       'Data Scientist/Machine Learning', 'Human Factors Scientist',
       'Business Intelligence Analyst I- Data Insights',
       'Data Scientist - Risk', 'Data Scientist-Human Resources',
       'Senior Research Statistician- Data Scientist', 'Data Engineer',
       'Associate Data Scientist', 'Business Intelligence Analyst',
       'Senior Analyst/Data Scientist', 'Data Analyst',
       'Machine Learning Engineer', 'Data Analyst I',
       'Scientist - Molecular Biology',
       'Computational Scientist, Machine Learning',
       'Senior Data Scientist', 'Jr. Data Engineer',
       'E-Commerce Data Analyst', 'Data Analytics Engineer',
       'Product Data Scient

In [8]:
# A base traz diferentes descrições para um mesmo cargo, o que prejudica a análise: Dessa forma não conseguimos agrupar vagas equivalentes.
# Como primeiro passo, vamos identificar os títulos de cargos mais frequentes no dataframe.
cargos = dfjobs['Job Title'].value_counts()
cargos_frequentes = cargos[cargos>3]
print(cargos_frequentes)

Job Title
Data Scientist                                                                                      337
Data Engineer                                                                                        26
Senior Data Scientist                                                                                19
Machine Learning Engineer                                                                            16
Data Analyst                                                                                         12
Senior Data Analyst                                                                                   6
Senior Data Engineer                                                                                  5
ENGINEER - COMPUTER SCIENTIST - RESEARCH COMPUTER SCIENTIST - SIGNAL PROCESSING - SAN ANTONIO OR      4
Data Science Software Engineer                                                                        4
Data Scientist - TS/SCI FSP or CI Required            

In [9]:
# Usando a função 'categoriza_cargo' vamos padronizar a descrição dos cargos identificados como mais frequentes.
def categoriza_cargo(title):
  t = title.lower()
  if 'data scientist' in t:
    return 'Data scientist'
  elif 'data engineer' in t:
    return 'Data engineer'
  elif 'machine learning engineer' in t:
    return 'Machine learning engineer'
  elif 'data analyst' in t:
    return 'Data analyst'
  elif 'software engineer' in t:  # "Data Science Software Engineer" cai aqui propositalmente
    return 'Software engineer'
  else:
    return 'outro'

# Criando uma nova coluna 'cargo_n' com as descrições padronizadas.
dfjobs['cargo_n'] = dfjobs['Job Title'].apply(categoriza_cargo)

In [10]:
# Iniciando com o tratamento dos salários
# Vamos extrair os salários mínimos e máximos de cada registro.
dfjobs['salario_min']= dfjobs['Salary Estimate'].str.split('-').str[0]
dfjobs['salario_max']= dfjobs['Salary Estimate'].str.split('-').str[1]

In [11]:
# Verificando se existem valores de Salário Mínimo que não estão na casa do milhares, para avaliar se podemos extrair os valores da coluna ignorando o "k".
print(dfjobs[~dfjobs['salario_min'].str.contains('K', na=False)]['salario_min'])

# Coma a série retorna vazia todos os valores estão na casa dos milhares.

Series([], Name: salario_min, dtype: object)


In [12]:
# Verificando se existem valores de Salário Mínimo que não estão na casa do milhares, para avaliar se podemos extrair os valores da coluna ignorando o "k".
print(dfjobs[~dfjobs['salario_max'].str.contains('K', na=False)]['salario_max'])

# Coma a série retorna vazia todos os valores estão na casa dos milhares.

Series([], Name: salario_max, dtype: object)


In [13]:
# Limpeza da coluna de salário para posterior conversão da variável em float.
# Removemos os símbolos de moeda ($), a abreviação de milhar (K) e o texto da fonte da estimativa, deixando apenas valores numéricos.
dfjobs['salario_min']= (
    dfjobs['salario_min']
    .str.replace('K','').str.replace('$','')
    .str.replace('K','').str.replace('$','')
    .str.strip()
)

dfjobs['salario_max']= (
    dfjobs['salario_max']
    .str.replace(r'\(.*\)','',regex=True) # Uso da Expressão Regular para remover o texto da fonte do salário
    .str.replace('K','').str.replace('$','')
    .str.replace('K','').str.replace('$','')
    .str.strip()
)


In [14]:
dfjobs['salario_med']= (dfjobs['salario_min'].astype(float) + dfjobs['salario_max'].astype(float))/2

In [15]:
dfjobs['Company Name'].unique()

array(['Healthfirst\n3.1', 'ManTech\n4.2', 'Analysis Group\n3.8',
       'INFICON\n3.5', 'Affinity Solutions\n2.9', 'HG Insights\n4.2',
       'Novartis\n3.9', 'iRobot\n3.5', 'Intuit - Data\n4.4',
       'XSELL Technologies\n3.6', 'Novetta\n4.5', '1904labs\n4.7',
       'PNNL\n3.7', 'Old World Industries\n3.1',
       'Mathematica Policy Research\n3.4',
       'Guzman & Griffin Technologies (GGTI)\n4.4',
       'Upside Business Travel\n4.1', 'Buckman\n3.5',
       'Insight Enterprises, Inc.\n4.2', 'Tower Health\n3.5',
       'Triplebyte\n3.2', 'PulsePoint\n4.3', 'Exponent\n3.5',
       'Guardian Life\n3.5',
       'Spectrum Communications and Consulting\n3.4',
       'Oversight Systems\n4.7', 'LSQ\n4.2',
       'MIT Lincoln Laboratory\n3.8', 'Kingfisher Systems\n4.5',
       'Formation\n2.8', 'Cohere Health\n5.0', 'Acuity Insurance\n4.8',
       'Chef\n3.6', 'Puget Sound Energy\n3.3', 'Sandhills Global\n2.7',
       'A Place for Mom\n2.7', 'Great-Circle Technologies\n2.2',
       'Edmu

In [16]:
# Removendo a quebra de linha dos valores da coluna 'Company Name'
dfjobs['Company Name']= dfjobs['Company Name'].str.split('\n').str[0]

In [17]:
dfjobs['Location'].unique()

array(['New York, NY', 'Chantilly, VA', 'Boston, MA', 'Newton, MA',
       'Santa Barbara, CA', 'Cambridge, MA', 'Bedford, MA',
       'San Diego, CA', 'Chicago, IL', 'Herndon, VA', 'Saint Louis, MO',
       'Richland, WA', 'Northbrook, IL', 'Washington, DC', 'Remote',
       'Memphis, TN', 'Plano, TX', 'West Grove, PA', 'Phoenix, AZ',
       'Appleton, WI', 'Atlanta, GA', 'Orlando, FL', 'Lexington, MA',
       'McLean, VA', 'San Francisco, CA', 'Sheboygan, WI',
       'United States', 'Bothell, WA', 'Lincoln, NE', 'Overland Park, KS',
       'Santa Monica, CA', 'Portsmouth, NH', 'Ewing, NJ',
       'South San Francisco, CA', 'Palo Alto, CA', 'Bellevue, WA',
       'New Orleans, LA', 'Akron, OH', 'Fort Wayne, IN', 'Woburn, MA',
       'Carson, CA', 'Coral Gables, FL', 'Santa Clara, CA',
       'Brisbane, CA', 'Winter Park, FL', 'Redwood City, CA',
       'Peoria, IL', 'Ipswich, MA', 'Carmel, IN', 'Emeryville, CA',
       'Gaithersburg, MD', 'Longmont, CO', 'Austin, TX', 'Yakima, WA',
 

In [18]:
dfjobs['Headquarters'].unique()

array(['New York, NY', 'Herndon, VA', 'Boston, MA',
       'Bad Ragaz, Switzerland', 'Santa Barbara, CA',
       'Basel, Switzerland', 'Bedford, MA', 'Mountain View, CA',
       'Chicago, IL', 'Mc Lean, VA', 'Saint Louis, MO', 'Richland, WA',
       'Northbrook, IL', 'Princeton, NJ', 'Mays Landing, NJ',
       'Washington, DC', 'Memphis, TN', 'Tempe, AZ', 'Reading, PA',
       'San Francisco, CA', 'Menlo Park, CA', 'Atlanta, GA',
       'Orlando, FL', 'Lexington, MA', 'Falls Church, VA',
       'Sheboygan, WI', 'Seattle, WA', 'Bellevue, WA', 'Lincoln, NE',
       'Chantilly, VA', 'Santa Monica, CA', 'Ewing, NJ',
       'South San Francisco, CA', 'Palo Alto, CA', 'Singapore, Singapore',
       'Cambridge, MA', 'OSAKA, Japan', 'Santa Clara, CA', 'Vienna, VA',
       'New Orleans, LA', 'Akron, OH', 'Zurich, Switzerland',
       'Woburn, MA', 'Carson, CA', 'Coral Gables, FL', 'San Ramon, CA',
       'Brisbane, CA', 'Winter Park, FL', 'San Rafael, CA',
       'Deerfield, IL', 'Ipswich, MA',

In [19]:
# Separando em novas colunas a cidade e o estado referentes a localização da vaga.
dfjobs['city_location']= dfjobs['Location'].str.split(',').str[0]
dfjobs['state_location']= dfjobs['Location'].str.split(',').str[1]

In [20]:
# Separando em novas colunas a cidade e o estado referentes a localização da sede da empresa.
dfjobs['city_Headquarters']= dfjobs['Headquarters'].str.split(',').str[0]
dfjobs['state_Headquarters']= dfjobs['Headquarters'].str.split(',').str[1]

In [21]:
colunas = ['city_location', 'state_location', 'city_Headquarters', 'state_Headquarters']

for col in colunas:
    dfjobs[col] = dfjobs[col].str.strip()

In [22]:
# Verificando se algum estado referente a localização da vaga não vem no padrão de escrita (sigla).
dfjobs[
    dfjobs['state_location'].str.strip()
    .str.len() > 2]

# Encontramos 'Anne Arundel' que faz parte do estado de Maryland (MD) no Estados Unidos.

,index,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,...,Competitors,Senioridae,cargo_n,salario_min,salario_max,salario_med,city_location,state_location,city_Headquarters,state_Headquarters
534,534,Data Scientist,$66K-$112K (Glassdoor est.),Job Description\nMUST HAVE SECRET CLEARANCE\n\...,3.4,"MILVETS Systems Technology, Inc.","Patuxent, Anne Arundel, MD","Orlando, FL",51 to 200 employees,1986.0,...,NaN,Unknown,Data scientist,66,112,89.0,Patuxent,Anne Arundel,Orlando,FL


In [23]:
# Vamos substituir 'Anne Arundel' na coluna 'state_location' pela a sigla do seu estado, utilizando o índice referente a esse registro.
dfjobs.loc[534, 'state_location'] = 'MD'

In [24]:
print(dfjobs.loc[534, 'state_location'])   # verificando a mudança

MD


In [25]:
# Existe ainda a possibilidade dos valores da coluna 'state_location' terem conteúdo numérico (ex.: "06"), o que passaria despercebido por uma verificação baseada apenas no tamanho.
# Utilizamos uma Expressão Regular para substituir esses casos por nulos.
mascara = dfjobs['state_location'].str.match(r'^\d+$', na=False) # A Expresão Regular captura células que vem com todo o conteúdo numérico, e o código da linha filtra as linhas em que essa condição é True.
dfjobs.loc[mascara, 'state_location'] = np.nan

In [26]:
# Agora vamos ralizar as mesmas análises para as colunas de localização da sede das empresas:
# Verificando se algum estado da Sede não vem no padrão de escrita (sigla).
dfjobs[
    dfjobs['state_Headquarters'].str.strip()
    .str.len() > 2]

,index,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,...,Competitors,Senioridae,cargo_n,salario_min,salario_max,salario_med,city_location,state_location,city_Headquarters,state_Headquarters
3,3,Data Scientist,$137K-$171K (Glassdoor est.),JOB DESCRIPTION:\n\nDo you have a passion for ...,3.5,INFICON,"Newton, MA","Bad Ragaz, Switzerland",501 to 1000 employees,2000.0,...,"MKS Instruments, Pfeiffer Vacuum, Agilent Tech...",Unknown,Data scientist,137,171,154.0,Newton,MA,Bad Ragaz,Switzerland
6,6,Data Scientist / Machine Learning Expert,$137K-$171K (Glassdoor est.),Posting Title\nData Scientist / Machine Learni...,3.9,Novartis,"Cambridge, MA","Basel, Switzerland",10000+ employees,1996.0,...,NaN,Unknown,Data scientist,137,171,154.0,Cambridge,MA,Basel,Switzerland
31,31,Data Scientist / Machine Learning Expert,$75K-$131K (Glassdoor est.),Posting Title\nData Scientist / Machine Learni...,3.9,Novartis,"Cambridge, MA","Basel, Switzerland",10000+ employees,1996.0,...,NaN,Unknown,Data scientist,75,131,103.0,Cambridge,MA,Basel,Switzerland
48,48,Data Scientist,$75K-$131K (Glassdoor est.),We are looking for Data Scientists who are int...,3.7,GovTech,"San Francisco, CA","Singapore, Singapore",1001 to 5000 employees,2016.0,...,NaN,Unknown,Data scientist,75,131,103.0,San Francisco,CA,Singapore,Singapore
51,51,Data Scientist,$75K-$131K (Glassdoor est.),"By clicking the Apply button, I understand tha...",3.7,Takeda,"Cambridge, MA","OSAKA, Japan",10000+ employees,1781.0,...,"Novartis, Baxter, Pfizer",Unknown,Data scientist,75,131,103.0,Cambridge,MA,OSAKA,Japan
60,60,Data Analyst,$75K-$131K (Glassdoor est.),About Swiss Re\n\nSwiss Re is one of the world...,3.8,Swiss Re,"Fort Wayne, IN","Zurich, Switzerland",10000+ employees,1863.0,...,"Munich Re, Hannover RE, SCOR",Unknown,Data analyst,75,131,103.0,Fort Wayne,IN,Zurich,Switzerland
80,80,"Real World Science, Data Scientist",$79K-$131K (Glassdoor est.),"Title: Real World Science, Data Scientist\nLoc...",4.0,AstraZeneca,"Gaithersburg, MD","Cambridge, United Kingdom",10000+ employees,1913.0,...,"Roche, GlaxoSmithKline, Novartis",Unknown,Data scientist,79,131,105.0,Gaithersburg,MD,Cambridge,United Kingdom
90,90,Patient Safety- Associate Data Scientist,$79K-$131K (Glassdoor est.),"Do you have expertise in, and passion for appl...",4.0,AstraZeneca,"Gaithersburg, MD","Cambridge, United Kingdom",10000+ employees,1913.0,...,"Roche, GlaxoSmithKline, Novartis",Junior,Data scientist,79,131,105.0,Gaithersburg,MD,Cambridge,United Kingdom
155,155,Sr Data Scientist,$90K-$109K (Glassdoor est.),bioMérieux Inc.\n\nSr Data Scientist\n\nSENIOR...,4.2,bioMérieux,"Saint Louis, MO","Marcy-l'Etoile, France",10000+ employees,1963.0,...,NaN,Senior,Data scientist,90,109,99.5,Saint Louis,MO,Marcy-l'Etoile,France
168,168,Data Engineer,$101K-$165K (Glassdoor est.),Job Number: 10202\nGroup: Cosma International\...,3.5,Magna International Inc.,"Birmingham, AL","Aurora, Canada",10000+ employees,1957.0,...,"Bosch, Lear Corporation, Faurecia",Unknown,Data engineer,101,165,133.0,Birmingham,AL,Aurora,Canada


Vamos que aplicando a mesa análise para a sede das empresas o resultado é diferente, alguns valores são de paises diferentes dos Estados Unidos.
Nesses casos na coluna 'state_Headquarters' não temos a sigla do estado e sim o nome do país.

In [27]:
# Aplicamos na coluna de estado das sedes ('state_Headquarters') a mesma máscara usada na coluna de estado das vagas, substituindo por nulos os valores totalmente numéricos.
mascara = dfjobs['state_Headquarters'].str.match(r'^\d+$', na=False)
dfjobs.loc[mascara, 'state_Headquarters'] = np.nan

In [28]:
# Vamos criar a coluna de país para diferenciar os registros onde a sede da empresa não fica nos Estados Unidos, como explicado na célula textual acima.
# A lógica: Quando o valor de state_Headquarters tem 2 caracteres, trata-se de uma sigla de estado americano, então o país é "United States of America"; caso contrário, o valor já representa o nome do país.

dfjobs['country_Headquarters'] = np.where(dfjobs['state_Headquarters'].str.len() == 2, 'United States of America', dfjobs['state_Headquarters']) # Criando a coluna de país

In [29]:
# Vamos replicar a mesma lógica para a localização da vaga por precaução.
dfjobs['country_location'] = np.where(dfjobs['state_location'].str.len() == 2, 'United States of America', dfjobs['state_location']) # Criando a coluna de país

In [30]:
# Identificando vagas remotas
dfjobs['remoto?'] = dfjobs['city_location'] == 'Remote'

In [31]:
dfjobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 672 entries, 0 to 671
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   index                 672 non-null    int64  
 1   Job Title             672 non-null    object 
 2   Salary Estimate       672 non-null    object 
 3   Job Description       672 non-null    object 
 4   Rating                622 non-null    float64
 5   Company Name          672 non-null    object 
 6   Location              672 non-null    object 
 7   Headquarters          641 non-null    object 
 8   Size                  645 non-null    object 
 9   Founded               554 non-null    float64
 10  Type of ownership     645 non-null    object 
 11  Industry              601 non-null    object 
 12  Sector                601 non-null    object 
 13  Revenue               645 non-null    object 
 14  Competitors           171 non-null    object 
 15  Senioridae            6

In [32]:
dfjobs=dfjobs.astype({
    'salario_min': 'float',
    'salario_max': 'float',
    'salario_med': 'float'
})

In [33]:
dfjobs[['salario_min','salario_max','salario_med']].describe()

,salario_min,salario_max,salario_med
count,672.000000,672.000000,672.000000
mean,99.196429,148.130952,123.663690
std,33.009958,48.035110,39.580268
min,31.000000,56.000000,43.500000
25%,79.000000,119.000000,103.000000
50%,91.000000,133.000000,114.000000
75%,122.000000,165.000000,136.500000
max,212.000000,331.000000,271.500000


In [34]:
pd.set_option('display.max_columns', None)
dfjobs.head(2)

,index,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,Senioridae,cargo_n,salario_min,salario_max,salario_med,city_location,state_location,city_Headquarters,state_Headquarters,country_Headquarters,country_location,remoto?
0,0,Sr Data Scientist,$137K-$171K (Glassdoor est.),Description\n\nThe Senior Data Scientist is re...,3.1,Healthfirst,"New York, NY","New York, NY",1001 to 5000 employees,1993.0,Nonprofit Organization,Insurance Carriers,Insurance,Unknown / Non-Applicable,"EmblemHealth, UnitedHealth Group, Aetna",Senior,Data scientist,137.0,171.0,154.0,New York,NY,New York,NY,United States of America,United States of America,False
1,1,Data Scientist,$137K-$171K (Glassdoor est.),"Secure our Nation, Ignite your Future\n\nJoin ...",4.2,ManTech,"Chantilly, VA","Herndon, VA",5001 to 10000 employees,1968.0,Company - Public,Research & Development,Business Services,$1 to $2 billion (USD),NaN,Unknown,Data scientist,137.0,171.0,154.0,Chantilly,VA,Herndon,VA,United States of America,United States of America,False


In [35]:
dfjobs['Size'].unique()

array(['1001 to 5000 employees', '5001 to 10000 employees',
       '501 to 1000 employees', '51 to 200 employees', '10000+ employees',
       '201 to 500 employees', '1 to 50 employees', nan, 'Unknown'],
      dtype=object)

In [36]:
# Vamos agrupar as faixas de número de funcionários em categorias de porte da empresa (Pequeno, Médio, Grande e Corporação), reduzindo as 7 faixas originais a 4 grupos mais fáceis de interpretar e visualizar.

categorias_tamanho = {
    '1 to 50 employees': 'Pequeno',
    '51 to 200 employees': 'Pequeno',
    '201 to 500 employees': 'Médio',
    '501 to 1000 employees': 'Médio',
    '1001 to 5000 employees': 'Grande',
    '5001 to 10000 employees': 'Grande',
    '10000+ employees': 'Corporação'
}

dfjobs['categoria_tamanho'] = dfjobs['Size'].map(categorias_tamanho)

In [37]:
# Criando o dataframe final somente com as colunas de interesse.

dffinal=dfjobs[['Rating', 'Company Name', 'Size', 'Founded', 'Type of ownership', 'Industry', 'Sector', 'Revenue', 'Senioridae', 'cargo_n', 'salario_med', 'city_location', 'state_location','country_location', 'city_Headquarters', 'country_Headquarters', 'categoria_tamanho', 'remoto?']].copy()

# Não vamos manter a variavél state_Headquarters porque tem paises misturados com estados.

In [38]:
dffinal = dffinal.rename(columns={
    'Rating': 'avaliacao',
    'Company Name': 'empresa',
    'Size': 'tamanho',
    'Founded': 'ano_fundacao',
    'Type of ownership': 'tipo_propriedade',
    'Industry': 'industria',
    'Sector': 'setor',
    'Revenue': 'receita',
    'Senioridae': 'senioridade',
    'cargo_n': 'cargo',
    'salario_med': 'salario_medio',
    'city_location': 'cidade_vaga',
    'state_location': 'estado_vaga',
    'country_location': 'pais_vaga',
    'city_Headquarters': 'cidade_sede',
    'country_Headquarters': 'pais_sede',
    'categoria_tamanho': 'escala_tamanho',
    'remoto?': 'remoto'
})

In [39]:
dffinal

,avaliacao,empresa,tamanho,ano_fundacao,tipo_propriedade,industria,setor,receita,senioridade,cargo,salario_medio,cidade_vaga,estado_vaga,pais_vaga,cidade_sede,pais_sede,escala_tamanho,remoto
0,3.1,Healthfirst,1001 to 5000 employees,1993.0,Nonprofit Organization,Insurance Carriers,Insurance,Unknown / Non-Applicable,Senior,Data scientist,154.0,New York,NY,United States of America,New York,United States of America,Grande,False
1,4.2,ManTech,5001 to 10000 employees,1968.0,Company - Public,Research & Development,Business Services,$1 to $2 billion (USD),Unknown,Data scientist,154.0,Chantilly,VA,United States of America,Herndon,United States of America,Grande,False
2,3.8,Analysis Group,1001 to 5000 employees,1981.0,Private Practice / Firm,Consulting,Business Services,$100 to $500 million (USD),Unknown,Data scientist,154.0,Boston,MA,United States of America,Boston,United States of America,Grande,False
3,3.5,INFICON,501 to 1000 employees,2000.0,Company - Public,Electrical & Electronic Manufacturing,Manufacturing,$100 to $500 million (USD),Unknown,Data scientist,154.0,Newton,MA,United States of America,Bad Ragaz,Switzerland,Médio,False
4,2.9,Affinity Solutions,51 to 200 employees,1998.0,Company - Private,Advertising & Marketing,Business Services,Unknown / Non-Applicable,Unknown,Data scientist,154.0,New York,NY,United States of America,New York,United States of America,Pequeno,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,3.6,TRANZACT,1001 to 5000 employees,1989.0,Company - Private,Advertising & Marketing,Business Services,Unknown / Non-Applicable,Unknown,Data scientist,136.0,Fort Lee,NJ,United States of America,Fort Lee,United States of America,Grande,False
668,NaN,JKGT,NaN,NaN,NaN,NaN,NaN,NaN,Unknown,Data scientist,136.0,San Francisco,CA,United States of America,NaN,NaN,NaN,False
669,NaN,AccessHope,NaN,NaN,NaN,NaN,NaN,NaN,Unknown,Data scientist,136.0,Irwindale,CA,United States of America,NaN,NaN,NaN,False
670,5.0,ChaTeck Incorporated,1 to 50 employees,NaN,Company - Private,Advertising & Marketing,Business Services,$1 to $5 million (USD),Unknown,Data scientist,136.0,San Francisco,CA,United States of America,Santa Clara,United States of America,Pequeno,False


In [40]:
# Importamos uma base auxiliar do GitHub (repositório google/dspl) contendo as coordenadas centrais (latitude e longitude) de cada estado dos Estados Unidos.
# Essas coordenadas serão associadas ao estado de cada vaga, permitindo posicionar os registros no mapa durante a etapa de visualização.

url = 'https://raw.githubusercontent.com/google/dspl/master/samples/google/canonical/states.csv'
df_coordenadas = pd.read_csv(url)
print(df_coordenadas.head())

  state   latitude   longitude        name
0    AK  63.588753 -154.493062      Alaska
1    AL  32.318231  -86.902298     Alabama
2    AR  35.201050  -91.831833    Arkansas
3    AZ  34.048928 -111.093731     Arizona
4    CA  36.778261 -119.417932  California


In [41]:
df_coordenadas['state'] = df_coordenadas['state'].str.strip()

In [42]:
# Unimos a base de vagas (dffinal) com a base de coordenadas, associando cada vaga à latitude e longitude do seu estado.
df_vagas = pd.merge(dffinal, df_coordenadas, left_on='estado_vaga', right_on='state', how='left')

In [43]:
df_vagas = df_vagas.drop(columns=['state'])
df_vagas = df_vagas.rename(columns={'name': 'nome_estado_vaga'})

In [44]:
# Exportamos a base final, já limpa e tratada, para um arquivo CSV.
df_vagas.to_csv('glassdoor_vagas_limpo.csv', index=False)